# Weekly Retail KPI — Exploratory Data Analysis
**Dataset:** 10 stores × 2 years (2024–2025) × ~50 weeks = 1,000 rows  
**Framework:** Schema validation → KPI engineering → Profiling → Benchmarking → YoY → Seasonality → Driver analysis → Segmentation → Anomaly detection → Synthesis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
})

PALETTE = sns.color_palette('tab10', 10)
DATA_PATH = Path(r'C:\Users\LENOVO\Desktop\Yoobic AI Technical PM USE CASE\practical-test-dataset-weekly-kpi.xlsx')

---
## Section 1 — Data Loading & Schema Validation
> **Guardrail:** Do not assume balanced coverage, zero missingness, or clean types. Validate the business key before computing any KPI.

In [ ]:
df = pd.read_excel(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
print('=' * 58)
print('SCHEMA & DTYPE VALIDATION')
print('=' * 58)

expected_dtypes = {
    'Store Name': 'object',
    'Year': 'int64',
    'Week': 'int64',
    'traffic': 'int64',
    'gross_transactions': 'int64',
    'gross_quantity': 'int64',
    'net_sales': 'float64',
}

for col, exp in expected_dtypes.items():
    actual = str(df[col].dtype)
    status = '✓' if actual == exp else f'✗ got {actual}'
    print(f'  {col:<22} expected={exp:<8}  {status}')

print(f'\nNull counts per column:')
print(df.isnull().sum().to_string())
print(f'\nYears present : {sorted(df["Year"].unique())}')
print(f'Week range     : {df["Week"].min()} – {df["Week"].max()}')
print(f'Unique stores  : {df["Store Name"].nunique()}')

In [ ]:
print('=' * 58)
print('BUSINESS KEY UNIQUENESS & COMPLETENESS')
print('=' * 58)

n_dups = df.duplicated(subset=['Store Name', 'Year', 'Week']).sum()
print(f'Duplicate (Store, Year, Week) rows: {n_dups}')

coverage = (
    df.groupby(['Store Name', 'Year'])['Week']
    .agg(rows='count', wk_min='min', wk_max='max', unique_wks='nunique')
)
print(f'\nWeek coverage per store-year:')
display(coverage)

all_combos = pd.MultiIndex.from_product(
    [df['Store Name'].unique(), df['Year'].unique(), range(1, df['Week'].max() + 1)],
    names=['Store Name', 'Year', 'Week']
)
actual_combos = pd.MultiIndex.from_frame(df[['Store Name', 'Year', 'Week']])
missing = all_combos.difference(actual_combos)
print(f'\nMissing store-year-week combinations: {len(missing)}')
if len(missing) > 0:
    display(pd.DataFrame(list(missing), columns=['Store Name', 'Year', 'Week']).head(20))

In [ ]:
print('=' * 58)
print('DOMAIN RULES VALIDATION')
print('=' * 58)

checks = {
    'traffic >= 0':                    (df['traffic'] < 0).sum(),
    'gross_transactions >= 0':         (df['gross_transactions'] < 0).sum(),
    'gross_quantity >= 0':             (df['gross_quantity'] < 0).sum(),
    'net_sales >= 0':                  (df['net_sales'] < 0).sum(),
    'transactions <= traffic':         (df['gross_transactions'] > df['traffic']).sum(),
    'quantity >= transactions':        (df['gross_quantity'] < df['gross_transactions']).sum(),
    'sales>0 when transactions==0':    ((df['gross_transactions'] == 0) & (df['net_sales'] > 0)).sum(),
    'zero traffic rows':               (df['traffic'] == 0).sum(),
    'zero transactions rows':          (df['gross_transactions'] == 0).sum(),
}

for rule, count in checks.items():
    flag = '✓ 0 violations' if count == 0 else f'⚠  {count} violations'
    print(f'  {rule:<40}  {flag}')

---
## Section 2 — Derived KPI Engineering
**Sales decomposition identity:**  
`net_sales = traffic × conversion_rate × units_per_txn × avg_selling_price`

All analysis is anchored to this tree — not to isolated metrics.

In [ ]:
df = df.copy()

safe_div = lambda num, den: np.where(den > 0, num / den, np.nan)

df['conversion_rate']     = safe_div(df['gross_transactions'], df['traffic'])
df['units_per_txn']       = safe_div(df['gross_quantity'],      df['gross_transactions'])
df['avg_txn_value']       = safe_div(df['net_sales'],           df['gross_transactions'])
df['avg_selling_price']   = safe_div(df['net_sales'],           df['gross_quantity'])
df['revenue_per_visitor'] = safe_div(df['net_sales'],           df['traffic'])

df['kpi_tree_reconstituted'] = (
    df['traffic'] *
    df['conversion_rate'] *
    df['units_per_txn'] *
    df['avg_selling_price']
)
max_err = ((df['net_sales'] - df['kpi_tree_reconstituted']).abs() /
           df['net_sales'].replace(0, np.nan)).max()
print(f'KPI tree reconstitution max relative error: {max_err:.2e}  (should be ≈ 0)')

RAW_KPIS     = ['traffic', 'gross_transactions', 'gross_quantity', 'net_sales']
DERIVED_KPIS = ['conversion_rate', 'units_per_txn', 'avg_txn_value', 'avg_selling_price', 'revenue_per_visitor']
ALL_KPIS     = RAW_KPIS + DERIVED_KPIS

STORE_ORDER  = df.groupby('Store Name')['net_sales'].sum().sort_values(ascending=False).index.tolist()
STORE_COLORS = {s: PALETTE[i] for i, s in enumerate(STORE_ORDER)}

print(f'\nStore order (by total net_sales): {STORE_ORDER}')

In [ ]:
kpi_dict = pd.DataFrame({
    'KPI': ['conversion_rate', 'units_per_txn', 'avg_txn_value', 'avg_selling_price', 'revenue_per_visitor'],
    'Formula': [
        'gross_transactions / traffic',
        'gross_quantity / gross_transactions',
        'net_sales / gross_transactions',
        'net_sales / gross_quantity',
        'net_sales / traffic',
    ],
    'Business Question': [
        'What % of visitors actually buy?',
        'How many items per basket?',
        'How much does each ticket cost?',
        'Implied price point per unit sold',
        'How much revenue does each visitor generate?',
    ],
    'Denominator Caveat': [
        'NaN when traffic = 0',
        'NaN when transactions = 0',
        'NaN when transactions = 0',
        'NaN when quantity = 0',
        'NaN when traffic = 0',
    ]
})
display(kpi_dict)

In [ ]:
df[['Store Name', 'Year', 'Week'] + DERIVED_KPIS].head(10)

---
## Section 3 — Descriptive Profiling
Robust statistics first, then shape. Using median/IQR over mean/std to avoid distortion from outlier weeks.

In [ ]:
def robust_profile(data, cols):
    rows = []
    for c in cols:
        s = data[c].dropna()
        rows.append({
            'metric': c,
            'count':  len(s),
            'median': s.median(),
            'IQR':    s.quantile(0.75) - s.quantile(0.25),
            'P10':    s.quantile(0.10),
            'P90':    s.quantile(0.90),
            'CV%':    round(s.std() / s.mean() * 100, 1) if s.mean() != 0 else np.nan,
            'skew':   round(s.skew(), 2),
        })
    return pd.DataFrame(rows).set_index('metric').round(3)

print('── Weekly observation-level profile ──')
display(robust_profile(df, ALL_KPIS))

agg_sy = df.groupby(['Store Name', 'Year'])[RAW_KPIS].sum()
agg_sy['conversion_rate']     = agg_sy['gross_transactions'] / agg_sy['traffic']
agg_sy['units_per_txn']       = agg_sy['gross_quantity']      / agg_sy['gross_transactions']
agg_sy['avg_txn_value']       = agg_sy['net_sales']           / agg_sy['gross_transactions']
agg_sy['avg_selling_price']   = agg_sy['net_sales']           / agg_sy['gross_quantity']
agg_sy['revenue_per_visitor'] = agg_sy['net_sales']           / agg_sy['traffic']

print('\n── Store-year aggregated profile ──')
display(robust_profile(agg_sy, ALL_KPIS))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, kpi in enumerate(ALL_KPIS):
    ax = axes[i]
    sns.histplot(df[kpi].dropna(), kde=True, ax=ax, color=PALETTE[i % 10], bins=30)
    ax.set_title(kpi.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.yaxis.set_visible(False)

plt.suptitle('KPI Distributions — Weekly Observations', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, kpi in enumerate(ALL_KPIS):
    ax = axes[i]
    sns.boxplot(
        data=df, x='Store Name', y=kpi, order=STORE_ORDER,
        palette='tab10', linewidth=0.8,
        flierprops={'markersize': 3, 'alpha': 0.5}, ax=ax
    )
    ax.set_title(kpi.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('KPI Distributions by Store (Weekly Grain)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Section 4 — Store Benchmarking: Scale vs Efficiency
Benchmarking is split into two dimensions to avoid conflating size with performance quality.

In [ ]:
store_yr = df.groupby(['Store Name', 'Year'])[RAW_KPIS].sum().reset_index()
store_yr['conversion_rate']     = store_yr['gross_transactions'] / store_yr['traffic']
store_yr['units_per_txn']       = store_yr['gross_quantity']      / store_yr['gross_transactions']
store_yr['avg_txn_value']       = store_yr['net_sales']           / store_yr['gross_transactions']
store_yr['avg_selling_price']   = store_yr['net_sales']           / store_yr['gross_quantity']
store_yr['revenue_per_visitor'] = store_yr['net_sales']           / store_yr['traffic']

store_yr.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
scale_kpis  = ['net_sales', 'traffic', 'gross_transactions', 'gross_quantity']
scale_titles = ['Total Net Sales', 'Total Traffic', 'Total Gross Transactions', 'Total Gross Quantity']

x = np.arange(len(STORE_ORDER))
w = 0.35

for ax, kpi, title in zip(axes.flatten(), scale_kpis, scale_titles):
    d24 = store_yr[store_yr['Year'] == 2024].set_index('Store Name').loc[STORE_ORDER, kpi]
    d25 = store_yr[store_yr['Year'] == 2025].set_index('Store Name').loc[STORE_ORDER, kpi]

    ax.bar(x - w/2, d24.values, w, label='2024', color='#4C72B0', alpha=0.85)
    ax.bar(x + w/2, d25.values, w, label='2025', color='#DD8452', alpha=0.85)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(STORE_ORDER, rotation=45, ha='right')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
    ax.legend()

plt.suptitle('Scale Benchmarking: 2024 vs 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
eff_kpis   = ['conversion_rate', 'units_per_txn', 'avg_txn_value', 'revenue_per_visitor']
eff_titles = ['Conversion Rate', 'Units Per Transaction', 'Avg Transaction Value (ATV)', 'Revenue Per Visitor']

for ax, kpi, title in zip(axes.flatten(), eff_kpis, eff_titles):
    d24 = store_yr[store_yr['Year'] == 2024].set_index('Store Name').loc[STORE_ORDER, kpi]
    d25 = store_yr[store_yr['Year'] == 2025].set_index('Store Name').loc[STORE_ORDER, kpi]

    ax.bar(x - w/2, d24.values, w, label='2024', color='#4C72B0', alpha=0.85)
    ax.bar(x + w/2, d25.values, w, label='2025', color='#DD8452', alpha=0.85)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(STORE_ORDER, rotation=45, ha='right')
    ax.legend()

plt.suptitle('Efficiency Benchmarking: 2024 vs 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
store_total = df.groupby('Store Name')[RAW_KPIS].sum()
store_total['conversion_rate']     = store_total['gross_transactions'] / store_total['traffic']
store_total['units_per_txn']       = store_total['gross_quantity']      / store_total['gross_transactions']
store_total['avg_txn_value']       = store_total['net_sales']           / store_total['gross_transactions']
store_total['avg_selling_price']   = store_total['net_sales']           / store_total['gross_quantity']
store_total['revenue_per_visitor'] = store_total['net_sales']           / store_total['traffic']

hm_kpis = ['net_sales', 'traffic', 'conversion_rate', 'avg_txn_value',
           'units_per_txn', 'revenue_per_visitor', 'avg_selling_price']

hm_raw  = store_total[hm_kpis].loc[STORE_ORDER]
hm_z    = (hm_raw - hm_raw.mean()) / hm_raw.std()

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    hm_z, annot=hm_raw.round(2), fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    annot_kws={'size': 8}, ax=ax
)
ax.set_title(
    'Store × KPI Performance Heatmap\n'
    'Color = Z-score (green=above avg)  |  Annotation = Raw value',
    fontsize=12
)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

---
## Section 5 — Year-over-Year Analysis
YoY % change per KPI per store, followed by a driver decomposition to explain *why* net_sales changed.

In [ ]:
yoy_kpis = ['net_sales', 'traffic', 'conversion_rate', 'units_per_txn', 'avg_txn_value', 'revenue_per_visitor']

pivot_24 = store_yr[store_yr['Year'] == 2024].set_index('Store Name')[yoy_kpis]
pivot_25 = store_yr[store_yr['Year'] == 2025].set_index('Store Name')[yoy_kpis]

yoy_pct = ((pivot_25 - pivot_24) / pivot_24 * 100).round(1)
yoy_pct = yoy_pct.loc[STORE_ORDER]

print('YoY % Change (2024 → 2025):')
display(yoy_pct.style.background_gradient(cmap='RdYlGn', axis=0).format('{:+.1f}%'))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, metric in enumerate(yoy_kpis):
    ax = axes[i]
    vals   = yoy_pct[metric].values
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in vals]

    ax.barh(STORE_ORDER, vals, color=colors, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(metric.replace('_', ' ').title() + '  YoY %')
    ax.set_xlabel('% Change')

    for j, v in enumerate(vals):
        offset = 0.4 if v >= 0 else -0.4
        ha = 'left' if v >= 0 else 'right'
        ax.text(v + offset, j, f'{v:+.1f}%', va='center', ha=ha, fontsize=8)

plt.suptitle('Year-over-Year Performance Change by Store (2024 → 2025)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print('Net Sales YoY Driver Decomposition')
print('net_sales = traffic × conversion_rate × units_per_txn × avg_selling_price')
print()

decomp_rows = []
for store in STORE_ORDER:
    d24 = store_yr[(store_yr['Store Name'] == store) & (store_yr['Year'] == 2024)].iloc[0]
    d25 = store_yr[(store_yr['Store Name'] == store) & (store_yr['Year'] == 2025)].iloc[0]

    def pct(v25, v24):
        return round((v25 - v24) / v24 * 100, 1) if v24 != 0 else np.nan

    asp24 = d24['net_sales'] / d24['gross_quantity']
    asp25 = d25['net_sales'] / d25['gross_quantity']
    cr24  = d24['gross_transactions'] / d24['traffic']
    cr25  = d25['gross_transactions'] / d25['traffic']
    upt24 = d24['gross_quantity'] / d24['gross_transactions']
    upt25 = d25['gross_quantity'] / d25['gross_transactions']

    decomp_rows.append({
        'Store':           store,
        'Traffic %':       pct(d25['traffic'],        d24['traffic']),
        'Conv Rate %':     pct(cr25,                  cr24),
        'UPT %':           pct(upt25,                 upt24),
        'ASP %':           pct(asp25,                 asp24),
        'Net Sales YoY %': pct(d25['net_sales'],      d24['net_sales']),
    })

decomp_df = pd.DataFrame(decomp_rows).set_index('Store')
display(decomp_df.style.background_gradient(cmap='RdYlGn', axis=0).format('{:+.1f}%'))

---
## Section 6 — Seasonality & Weekly Trends
Analyze within each year before pooling. Three questions: level shift, recurring seasonality, per-store volatility.

In [ ]:
weekly_avg = df.groupby(['Year', 'Week'])[ALL_KPIS].mean().reset_index()

trend_kpis   = ['net_sales', 'traffic', 'conversion_rate', 'avg_txn_value']
trend_titles = ['Avg Net Sales', 'Avg Traffic', 'Avg Conversion Rate', 'Avg Transaction Value']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, metric, title in zip(axes.flatten(), trend_kpis, trend_titles):
    for yr, color in zip([2024, 2025], ['#4C72B0', '#DD8452']):
        d = weekly_avg[weekly_avg['Year'] == yr].sort_values('Week')
        ax.plot(d['Week'], d[metric], label=str(yr), color=color, linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Week of Year')
    ax.legend()

plt.suptitle('Average Weekly Trends by Year (All Stores Combined)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(22, 11))

for ax, yr in zip(axes, [2024, 2025]):
    pivot = df[df['Year'] == yr].pivot_table(
        index='Store Name', columns='Week', values='net_sales', aggfunc='mean'
    ).loc[STORE_ORDER]

    sns.heatmap(
        pivot, ax=ax, cmap='YlOrRd',
        linewidths=0.15, linecolor='white',
        cbar_kws={'shrink': 0.5, 'label': 'Avg Net Sales'}
    )
    ax.set_title(f'Net Sales by Store × Week — {yr}', fontsize=12)
    ax.set_xlabel('Week Number')
    ax.set_ylabel('')

plt.suptitle('Seasonality Heatmap: Net Sales (Store × Week)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
grand_weekly = df.groupby('Week')['net_sales'].mean().reset_index().rename(columns={'net_sales': 'avg_net_sales'})
top5    = grand_weekly.nlargest(5,  'avg_net_sales')
bottom5 = grand_weekly.nsmallest(5, 'avg_net_sales')

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(grand_weekly['Week'], grand_weekly['avg_net_sales'],
        color='#4C72B0', linewidth=1.5, label='Avg Net Sales')
ax.scatter(top5['Week'],    top5['avg_net_sales'],    color='green', s=90, zorder=5, label='Top 5 weeks')
ax.scatter(bottom5['Week'], bottom5['avg_net_sales'], color='red',   s=90, zorder=5, label='Bottom 5 weeks')

for _, row in top5.iterrows():
    ax.annotate(f"Wk {int(row['Week'])}",
                xy=(row['Week'], row['avg_net_sales']),
                xytext=(0, 9), textcoords='offset points',
                ha='center', fontsize=8, color='green')
for _, row in bottom5.iterrows():
    ax.annotate(f"Wk {int(row['Week'])}",
                xy=(row['Week'], row['avg_net_sales']),
                xytext=(0, -14), textcoords='offset points',
                ha='center', fontsize=8, color='red')

ax.set_title('Average Weekly Net Sales (All Stores & Years) — Peak & Trough Weeks')
ax.set_xlabel('Week Number')
ax.set_ylabel('Avg Net Sales')
ax.legend()
plt.tight_layout()
plt.show()

print('Top 5 peak weeks:')
print(top5.to_string(index=False))
print('\nBottom 5 trough weeks:')
print(bottom5.to_string(index=False))

In [ ]:
volatility = (
    df.groupby(['Store Name', 'Year'])['net_sales']
    .agg(median='median', std='std', cv=lambda s: s.std() / s.mean() * 100)
    .round(2)
)
print('Weekly Net Sales Volatility per Store-Year (CV% = std/mean × 100):')
display(volatility.reset_index().pivot(index='Store Name', columns='Year', values='cv')
        .loc[STORE_ORDER]
        .style.background_gradient(cmap='Reds').format('{:.1f}%'))

---
## Section 7 — Driver Analysis (KPI Tree)
Relationships examined along the KPI tree paths, not as a generic correlation matrix.  
Avoids circular reasoning from ratio metrics sharing numerators/denominators with raw metrics.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

pairs = [
    ('traffic',             'gross_transactions', 'Traffic → Transactions'),
    ('gross_transactions',  'net_sales',          'Transactions → Net Sales'),
    ('gross_quantity',      'net_sales',          'Quantity → Net Sales'),
]

for ax, (x_col, y_col, title) in zip(axes, pairs):
    for store in STORE_ORDER:
        mask = df['Store Name'] == store
        ax.scatter(df.loc[mask, x_col], df.loc[mask, y_col],
                   color=STORE_COLORS[store], alpha=0.35, s=14, label=store)

    ax.set_xlabel(x_col.replace('_', ' ').title())
    ax.set_ylabel(y_col.replace('_', ' ').title())
    ax.set_title(title)

handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=STORE_COLORS[s], markersize=8, label=s)
    for s in STORE_ORDER
]
axes[2].legend(handles=handles, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

plt.suptitle('KPI Tree Driver Analysis (Weekly Observations)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
corr_rows = []
for store in STORE_ORDER:
    sub = df[df['Store Name'] == store]
    for x_col in ['traffic', 'conversion_rate', 'avg_txn_value', 'units_per_txn']:
        r = sub[x_col].corr(sub['net_sales'])
        corr_rows.append({'Store': store, 'Driver': x_col, 'r_with_net_sales': round(r, 3)})

corr_pivot = (
    pd.DataFrame(corr_rows)
    .pivot(index='Store', columns='Driver', values='r_with_net_sales')
    .loc[STORE_ORDER]
)

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(
    corr_pivot, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title(
    'Per-Store Pearson Correlation: Driver vs Net Sales\n'
    '(raw-metric drivers only — avoids circular numerator/denominator bias)',
    fontsize=11
)
plt.tight_layout()
plt.show()

---
## Section 8 — Store Segmentation / Quadrant Analysis
Stores grouped by strategic profile — generates actionable archetypes, not just rankings.

In [ ]:
store_med = df.groupby('Store Name')[['traffic', 'conversion_rate', 'avg_txn_value',
                                       'units_per_txn', 'net_sales', 'revenue_per_visitor']].median()
store_med = store_med.loc[STORE_ORDER]
display(store_med.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

x_vals = store_med['traffic']
y_vals = store_med['conversion_rate']
sizes  = (store_med['net_sales'] / store_med['net_sales'].max() * 900).values

ax.scatter(x_vals, y_vals, s=sizes, alpha=0.75,
           c=[PALETTE[i] for i in range(len(STORE_ORDER))],
           edgecolors='black', linewidths=0.8)

for i, store in enumerate(STORE_ORDER):
    ax.annotate(store, (x_vals.iloc[i], y_vals.iloc[i]),
                textcoords='offset points', xytext=(7, 5), fontsize=9, fontweight='bold')

med_x, med_y = x_vals.median(), y_vals.median()
ax.axvline(med_x, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.axhline(med_y, color='gray', linestyle='--', linewidth=1, alpha=0.6)

quadrant_labels = [
    (x_vals.min(), y_vals.max(), 'left',  'top',    'Low Traffic\nHigh Conv.\n(Efficient Niche)'),
    (x_vals.max(), y_vals.max(), 'right', 'top',    'High Traffic\nHigh Conv.\n(Star Store)'),
    (x_vals.min(), y_vals.min(), 'left',  'bottom', 'Low Traffic\nLow Conv.\n(Underperformer)'),
    (x_vals.max(), y_vals.min(), 'right', 'bottom', 'High Traffic\nLow Conv.\n(Leaky Funnel)'),
]
for qx, qy, ha, va, label in quadrant_labels:
    ax.text(qx, qy, label, ha=ha, va=va, fontsize=8, color='gray', style='italic')

ax.set_xlabel('Median Weekly Traffic')
ax.set_ylabel('Median Conversion Rate')
ax.set_title(
    'Store Segmentation: Traffic vs Conversion Rate\n'
    'Bubble size = Median Net Sales | Dashed lines = portfolio medians',
    fontsize=12
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

x_vals = store_med['avg_txn_value']
y_vals = store_med['units_per_txn']

ax.scatter(x_vals, y_vals, s=400, alpha=0.8,
           c=[PALETTE[i] for i in range(len(STORE_ORDER))],
           edgecolors='black', linewidths=0.8)

for i, store in enumerate(STORE_ORDER):
    ax.annotate(store, (x_vals.iloc[i], y_vals.iloc[i]),
                textcoords='offset points', xytext=(7, 5), fontsize=9, fontweight='bold')

ax.axvline(x_vals.median(), color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.axhline(y_vals.median(), color='gray', linestyle='--', linewidth=1, alpha=0.6)

ax.set_xlabel('Median Avg Transaction Value (ATV)')
ax.set_ylabel('Median Units Per Transaction (UPT)')
ax.set_title(
    'Basket Profile: ATV vs UPT\n'
    'Top-right = large high-value baskets  |  Bottom-left = small low-value baskets',
    fontsize=12
)
plt.tight_layout()
plt.show()

---
## Section 9 — Anomaly & Outlier Detection
IQR flagging is applied **within each store-year context** to account for natural scale differences.  
Using 2× IQR fence (tighter than default 1.5×) to surface only meaningful deviations.

In [ ]:
ANOMALY_KPIS = ['traffic', 'gross_transactions', 'net_sales', 'conversion_rate', 'avg_txn_value']
IQR_FENCE    = 2.0

flagged_parts = []

for (store, year), grp in df.groupby(['Store Name', 'Year']):
    for kpi in ANOMALY_KPIS:
        vals = grp[kpi].dropna()
        Q1, Q3 = vals.quantile(0.25), vals.quantile(0.75)
        IQR   = Q3 - Q1
        lo    = Q1 - IQR_FENCE * IQR
        hi    = Q3 + IQR_FENCE * IQR

        flags = grp[(grp[kpi] < lo) | (grp[kpi] > hi)].copy()
        if not flags.empty:
            flags['kpi_flagged']  = kpi
            flags['lower_fence']  = round(lo, 2)
            flags['upper_fence']  = round(hi, 2)
            flags['direction']    = flags[kpi].apply(lambda v: 'HIGH' if v > hi else 'LOW')
            flagged_parts.append(
                flags[['Store Name', 'Year', 'Week', 'kpi_flagged', kpi, 'lower_fence', 'upper_fence', 'direction']]
            )

anomaly_log = pd.concat(flagged_parts, ignore_index=True) if flagged_parts else pd.DataFrame()
print(f'Total anomaly flags ({IQR_FENCE}× IQR, per store-year context): {len(anomaly_log)}')
if not anomaly_log.empty:
    print(f'\nFlags by KPI:')
    print(anomaly_log.groupby(['kpi_flagged', 'direction']).size().to_string())
    print(f'\nFlags by Store:')
    print(anomaly_log.groupby('Store Name').size().sort_values(ascending=False).to_string())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes_flat = axes.flatten()

for ax, store in zip(axes_flat, STORE_ORDER):
    sub = df[df['Store Name'] == store].sort_values('Week')

    anom_store = (
        anomaly_log[(anomaly_log['Store Name'] == store) & (anomaly_log['kpi_flagged'] == 'net_sales')]
        if not anomaly_log.empty else pd.DataFrame()
    )

    for yr, color in zip([2024, 2025], ['#4C72B0', '#DD8452']):
        s_yr = sub[sub['Year'] == yr]
        ax.scatter(s_yr['Week'], s_yr['net_sales'], color=color, s=14, alpha=0.7, label=str(yr))

        if not anom_store.empty:
            anom_yr  = anom_store[anom_store['Year'] == yr]
            if not anom_yr.empty:
                mask = (sub['Year'] == yr) & (sub['Week'].isin(anom_yr['Week']))
                ax.scatter(sub.loc[mask, 'Week'], sub.loc[mask, 'net_sales'],
                           color='red', s=60, zorder=5, marker='x', linewidths=2)

    ax.set_title(store, fontsize=9)
    ax.set_xlabel('Week', fontsize=7)
    ax.tick_params(labelsize=7)

axes_flat[0].legend(fontsize=7)
plt.suptitle(
    f'Net Sales per Week by Store  |  Red × = Net Sales Anomaly ({IQR_FENCE}× IQR within Store-Year)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
if not anomaly_log.empty:
    print('Anomaly Log — Top 30 flags requiring business follow-up:')
    display(
        anomaly_log
        .sort_values(['Store Name', 'Year', 'Week'])
        .head(30)
        .reset_index(drop=True)
    )
else:
    print('No anomalies detected at the 2× IQR threshold.')

---
## Section 10 — Executive Synthesis
Answers the three core business questions from the KPI decomposition framework.

In [ ]:
print('=' * 68)
print('EXECUTIVE SYNTHESIS — WEEKLY RETAIL KPI EDA')
print('=' * 68)

# 1. Revenue concentration
rev_by_store = df.groupby('Store Name')['net_sales'].sum().sort_values(ascending=False)
top3_share   = rev_by_store.head(3).sum() / rev_by_store.sum() * 100
print(f'\n1. REVENUE CONCENTRATION')
print(f'   Top 3 stores: {rev_by_store.head(3).index.tolist()}')
print(f'   Share of total net sales: {top3_share:.1f}%')
print(f'   Lowest revenue store: {rev_by_store.index[-1]} ({rev_by_store.iloc[-1]:,.0f})')

# 2. YoY winners & losers
if 'net_sales' in yoy_pct.columns:
    ns_yoy = yoy_pct['net_sales']
    print(f'\n2. YEAR-OVER-YEAR NET SALES (2024 → 2025)')
    print(f'   Best growth:  {ns_yoy.idxmax()} ({ns_yoy.max():+.1f}%)')
    print(f'   Worst change: {ns_yoy.idxmin()} ({ns_yoy.min():+.1f}%)')
    grew = (ns_yoy > 0).sum()
    print(f'   {grew}/{len(ns_yoy)} stores grew net_sales YoY')

# 3. Efficiency spread
cr = store_med['conversion_rate']
atv = store_med['avg_txn_value']
print(f'\n3. EFFICIENCY GAPS')
print(f'   Conversion rate — best: {cr.idxmax()} ({cr.max():.1%})  worst: {cr.idxmin()} ({cr.min():.1%})')
print(f'   ATV             — best: {atv.idxmax()} ({atv.max():,.0f})  lowest: {atv.idxmin()} ({atv.min():,.0f})')
print(f'   Conversion gap of {(cr.max()-cr.min()):.1%} — {"significant" if cr.max()-cr.min() > 0.05 else "moderate"}')

# 4. YoY primary driver
if not decomp_df.empty:
    avg_drivers = decomp_df.mean()
    primary_driver = avg_drivers.drop('Net Sales YoY %').abs().idxmax()
    print(f'\n4. PRIMARY YoY DRIVER (portfolio average)')
    print(f'   Dominant factor: {primary_driver} (largest absolute avg % swing)')
    print(decomp_df.mean().round(1).to_string())

# 5. Anomaly summary
print(f'\n5. ANOMALY FLAGS ({IQR_FENCE}× IQR threshold)')
if not anomaly_log.empty:
    by_store = anomaly_log.groupby('Store Name').size().sort_values(ascending=False)
    print(f'   Total flags: {len(anomaly_log)}')
    print(f'   Most flagged store: {by_store.index[0]} ({by_store.iloc[0]} flags)')
    print(f'   Stores with zero flags: {(by_store == 0).sum() + (len(STORE_ORDER) - len(by_store))} / {len(STORE_ORDER)}')
else:
    print('   No anomalies detected.')

# 6. Open questions for next phase
print(f'\n6. OPEN QUESTIONS FOR NEXT PHASE')
print('   a) What drives the conversion rate gap — product mix, pricing, or operations?')
print('   b) Are anomalous weeks linked to promotions, closures, or data issues?')
print('   c) Does the YoY net_sales driver decompose differently across store archetypes?')
print('   d) Does seasonal timing differ across stores (simultaneous peak or staggered)?')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue share pie
rev_pct = (rev_by_store / rev_by_store.sum() * 100).round(1)
axes[0].pie(
    rev_pct.values,
    labels=rev_pct.index,
    autopct='%1.1f%%',
    colors=PALETTE,
    startangle=140,
    pctdistance=0.82,
)
axes[0].set_title('Revenue Share by Store (2024+2025 Combined)', fontsize=12)

# YoY net sales ranked bar
if 'net_sales' in yoy_pct.columns:
    ns_yoy_sorted = yoy_pct['net_sales'].sort_values()
    colors = ['#2ca02c' if v >= 0 else '#d62728' for v in ns_yoy_sorted]
    axes[1].barh(ns_yoy_sorted.index, ns_yoy_sorted.values, color=colors, alpha=0.85)
    axes[1].axvline(0, color='black', linewidth=0.8)
    axes[1].set_title('Net Sales YoY % Change — Ranked', fontsize=12)
    axes[1].set_xlabel('% Change (2024 → 2025)')
    for i, v in enumerate(ns_yoy_sorted.values):
        offset = 0.3 if v >= 0 else -0.3
        ha = 'left' if v >= 0 else 'right'
        axes[1].text(v + offset, i, f'{v:+.1f}%', va='center', ha=ha, fontsize=9)

plt.suptitle('Summary: Revenue Concentration & YoY Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()